# Chit 2 baseline: an autoencoder anomaly trigger · **FPGA-L1**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Train a quantized autoencoder on background events only, use its reconstruction error as the anomaly score, measure the signal efficiency (TPR) at the dataset's standard false-positive rate of 10⁻⁵ (~400 Hz of the 40 MHz collision rate), and check that the hls4ml emulation of the same network is bit-exact. For scale: the state of the art is ~4-5% on A→4ℓ; anything above 1% is good.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~3 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~2 min: it installs Brevitas and hls4ml) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["brevitas", "hls4ml==1.3.0", "quantizers<2", "qonnx", "onnxoptimizer", "onnx"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 0. Understand the data
Before any model, look at the data: what one sample is, what the classes look like, how the splits were made, and why the chit uses its metric.
This part only reads and plots (≲ 1 min). The model starts in Part 1.

### 0.1 The task in plain words
The LHC collides proton bunches 40 million times per second. Detectors cannot store all of it, so a fast **trigger** (FPGAs, a few µs) decides for every collision
("event") whether to keep it. Today's triggers keep events that look like physics we already know (e.g. two energetic muons). An **anomaly trigger** instead keeps events
that look *unlike* ordinary collisions, so it could catch new physics that nobody has predicted. It is trained on ordinary collisions only, with no example of the new physics.
**One decision of the model** = an anomaly score for one event; the trigger keeps the event if the score is above a threshold.

### 0.2 What one sample is
One event is what the trigger sees: up to 19 reconstructed **objects** in fixed slots: the missing transverse energy (MET, slot 0: energy that no detector saw,
e.g. neutrinos), up to 4 electrons, up to 4 muons and up to 10 jets (sprays of particles). Each object has 3 numbers: **pT** (momentum across the beam, in GeV),
**η** (direction along the beam: 0 = at right angles to the beam, ±4 = close to the beam) and **φ** (angle around the beam, in radians). So one event = 19 × 3 = **57 numbers**.
An empty slot is three zeros. MET has no η (always 0). Electrons are sorted by pT (highest first), and so are muons.
**Label** (val and test only): 0 = background (ordinary collisions), 1–4 = one of four simulated new-physics signals. Below: the first background event and the first
A→4ℓ event of `val`, slot by slot.

In [ ]:
import matplotlib.pyplot as plt
from capstone.data.adc2021 import load, SIGNALS, OBJECTS
X0, y0 = load("val")                              # 1M background events + the four signals (labels 1-4)
NAMES = ["background"] + SIGNALS
pt, eta, phi = X0[:, 0::3], X0[:, 1::3], X0[:, 2::3]          # (events, 19) each: slot 0 = MET, 1-4 electrons, 5-8 muons, 9-18 jets
print("one event =", X0.shape[1], "numbers = 19 objects × (pT, η, φ)")
b, s = np.where(y0 == 0)[0][0], np.where(y0 == 1)[0][0]       # the first background event and the first A→4ℓ event of val
print(f"\n{'slot':<6}{'background: pT [GeV]':>22}{'η':>7}{'φ':>7}   |{'A→4ℓ: pT [GeV]':>16}{'η':>7}{'φ':>7}")
for j, o in enumerate(OBJECTS):
    print(f"{o:<6}{pt[b, j]:>22.1f}{eta[b, j]:>7.2f}{phi[b, j]:>7.2f}   |{pt[s, j]:>16.1f}{eta[s, j]:>7.2f}{phi[s, j]:>7.2f}")

Most of the 57 numbers are zeros: the background event has only MET and one electron. In the A→4ℓ event, jet1 sits at the same (η, φ) as electron e0:
the trigger's jet finder also picks up the electron, so one particle can appear twice (about one electron in four has a jet within 0.2 in (η, φ)).
The A→4ℓ signal makes four leptons, but here only two are seen: the others flew outside the part of the detector the trigger sees, or had too little energy to be reconstructed.

In [ ]:
occ = pt != 0
n_lep, n_jet = occ[:, 1:9].sum(1), occ[:, 9:].sum(1)
lead_lep = pt[:, 1:9].max(1)
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
for k, name in enumerate(NAMES):
    m = y0 == k
    kw = dict(histtype="step", density=True, color=f"C{k}" if k else "k", lw=2 if k == 0 else 1.2, label=name)
    ax[0].hist(n_lep[m], np.arange(0, 9) - 0.5, **kw)
    ax[1].hist(n_jet[m], np.arange(0, 12) - 0.5, **kw)
    ax[2].hist(lead_lep[m], np.linspace(0, 500, 101), **kw)
    ax[3].hist(pt[m, 0], np.linspace(0, 500, 101), **kw)
for a, xl, t in zip(ax, ("number of leptons (electrons + muons)", "number of jets", "leading lepton pT [GeV]", "missing transverse energy [GeV]"),
                    ("Leptons per event", "Jets per event", "Highest lepton pT", "Missing energy")):
    a.set(xlabel=xl, ylabel="fraction of events (normalised)", yscale="log", title=t)
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
for k, name in enumerate(NAMES):
    m = y0 == k
    print(f"{name:<11} exactly 1 lepton {np.mean(n_lep[m] == 1):6.1%} | 3 or more leptons {np.mean(n_lep[m] >= 3):7.3%} | median MET {np.median(pt[m, 0]):5.1f} GeV")

**What to notice.** Background events almost always have **exactly one lepton** (99.3%); this data set was selected with one lepton above 23 GeV,
which is why no event has zero leptons and the highest-lepton pT starts at 23 GeV. A→4ℓ events are different mainly in the number of leptons (half have 3 or more,
against 0.07% of the background), and they have *less* missing energy than the background. The other three signals look much more like background:
mostly one lepton, more jets, and h±→τν has much more missing energy (neutrinos). Note the log scale: at a false-positive rate of 10⁻⁵, what matters is
the far tail of the background (for example, about 1 background event in 10,000 still has 4 leptons).

### 0.3 Look at it
Four background events (top) and four A→4ℓ events (bottom), drawn in the (η, φ) plane of the detector: each marker is one object, its size grows with pT,
and the dashed line shows the direction (φ) of the missing energy.

In [ ]:
from matplotlib.lines import Line2D
STYLE = ((slice(1, 5), "o", "C0", "electron"), (slice(5, 9), "s", "C3", "muon"), (slice(9, 19), "^", "0.5", "jet"))
fig, ax = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)
for row, k in enumerate((0, 1)):
    for a, i in zip(ax[row], np.where(y0 == k)[0][:4]):
        for sl, mk, c, lab in STYLE:
            on = occ[i, sl]
            a.scatter(eta[i, sl][on], phi[i, sl][on], s=8 * np.sqrt(pt[i, sl][on]) * 3, marker=mk, color=c, alpha=0.7, edgecolor="k", lw=0.4)
        a.axhline(phi[i, 0], color="C2", ls="--", lw=1.2)
        a.text(-4, phi[i, 0] + 0.1, f"MET {pt[i, 0]:.0f} GeV", color="C2", fontsize=8)
        a.set(xlim=(-4.2, 4.2), ylim=(-3.4, 3.4), title=f"{NAMES[k]}: {occ[i, 1:5].sum()} e, {occ[i, 5:9].sum()} μ, {occ[i, 9:].sum()} jets")
    ax[row, 0].set_ylabel("φ (angle around the beam) [rad]")
for a in ax[1]:
    a.set_xlabel("η (direction along the beam)")
ax[0, 3].legend([Line2D([], [], marker=mk, color=c, ls="", markersize=8) for _, mk, c, _ in STYLE] + [Line2D([], [], color="C2", ls="--")],
                [lab for *_, lab in STYLE] + ["MET direction"], fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

**What to notice.** Background events are nearly empty: one lepton, often no jet. A→4ℓ events have more objects, several leptons, and leptons that overlap with jets.
No single picture says "anomaly": the network must learn what ordinary looks like from 2 million background events, and flag what does not fit.

### 0.4 The numbers that matter

In [ ]:
X_tr0, _ = load("train"); X_te0, y_te0 = load("test")
print(f"{'split':<6}{'background':>12}" + "".join(f"{s:>10}" for s in SIGNALS))
print(f"{'train':<6}{len(X_tr0):>12,}" + "".join(f"{'-':>10}" for s in SIGNALS) + "   (no labels: background only)")
for name, y in (("val", y0), ("test", y_te0)):
    print(f"{name:<6}{np.sum(y == 0):>12,}" + "".join(f"{np.sum(y == k):>10,}" for k in range(1, 5)))
print(f"\nempty slots (zeros): {np.mean(X_tr0 == 0):.0%} of all training numbers")
for q, j in (("pT [GeV]", 0), ("η", 1), ("φ [rad]", 2)):
    v = X_tr0[:, j::3]; v = v[v != 0]
    print(f"{q:<9} non-empty values: {v.min():8.2f} to {v.max():8.2f}  (99.9% below {np.quantile(np.abs(v), 0.999):.1f})")
del X_tr0, X_te0

**Splits.** The 4,000,000 background events are shuffled once (fixed seed) and split 2M / 1M / 1M into train / val / test. Each signal is shuffled and split half / half
between val and test. **Train has background only and no labels**: the method must be unsupervised, as a real anomaly trigger would be.
You may use the signals in val to choose a design; report the final numbers on test. The signals are much rarer in reality than here; the counts are only for measuring efficiencies.
**Value ranges.** 88% of all input numbers are zeros (empty slots). pT spans 0 to ~2,000 GeV, but 99.9% of the values are below ~220 GeV; η is within ±4, φ within ±π.
So scale the inputs (the baseline divides pT, η and φ by a power of two each), using the training split only.

### 0.5 The metric, explained on the data
The trigger may keep only a fixed rate of background: here 10⁻⁵ of all events (~400 Hz out of 40 MHz), because that is what the readout can afford.
So the threshold is set by the background, and the question is **how much of each signal passes** at that threshold (true-positive rate at FPR = 10⁻⁵).
Accuracy would be useless: "keep nothing" is right for 99.999% of events. With 1,000,000 background events in test, only the 10 highest-scoring ones set the threshold,
so small differences between designs are noise: compare designs with several seeds. The plots show why A→4ℓ is the easiest signal: more leptons is a strong, simple clue.

In [ ]:
del X0, pt, eta, phi, occ

## 1. Data
Training is **background only, without labels**. Val and test mix 1M background events with four new-physics signals (label 1-4).

In [ ]:
from capstone.data.adc2021 import load, efficiency_at_fpr, SIGNALS, FPR
N_TRAIN = 50_000 if FAST else 500_000            # a subset of the 2M training events keeps training to a few minutes
X_tr, _ = load("train", max_items=N_TRAIN)
X_va, y_va = load("val")
X_te, y_te = load("test")
print("train", X_tr.shape, "| val", X_va.shape, "| test", X_te.shape)
for k, s in enumerate(SIGNALS, start=1):
    print(f"  test: {s:<6} {np.sum(y_te == k):>7,} events")
print(f"  test: background {np.sum(y_te == 0):,}")

**Input scaling.** pT runs to hundreds of GeV, η and φ to a few units. We divide each **quantity** (all pT's, all η's, all φ's) by one **power of two**
chosen from the training split: the smallest one above its 99.9th percentile over non-empty slots (most slots are empty, i.e. zero). Most inputs then land
in [-1, 1]. On an FPGA a division by a power of two is free (a bit shift).

In [ ]:
KIND = np.arange(57) % 3                         # 0 = pT, 1 = eta, 2 = phi (slot by slot)
def pow2_scale(v):
    v = np.abs(v[v != 0])
    return 2.0 ** np.ceil(np.log2(np.quantile(v, 0.999)))
SCALE = np.array([pow2_scale(X_tr[:, KIND == k]) for k in range(3)])[KIND]
print("scales (pT, eta, phi):", SCALE[:3])
sc = lambda X: (X / SCALE).astype(np.float32)
Xtr, Xva, Xte = sc(X_tr), sc(X_va), sc(X_te)

## 2. The model, costed before training
A dense autoencoder 57 → 32 → 8 → 32 → 57, every layer 8-bit (weights and inputs), built with `qat_mlp`: the only builder that can be synthesized for the FPGA chits.

In [ ]:
from common.compress import qat_mlp
SIZES = (57, 32, 8, 32, 57)
CFG = [(8, 8)] * 4                               # (weight bits, input bits) per layer
model = qat_mlp(CFG, SIZES)
p = profile(model, (57,))
print(f"params {p['params']:,} | MACs {p['macs']:,} | weights at 8 bits {p['weight_bytes']:,} B")

## 3. Train on background, score by reconstruction error
Each feature's squared error is weighted by 1 / (its variance on the training split). Without that, the large pT values dominate the loss and the
"anomaly score" degenerates into "total pT". The same weights are used in the score (constants: cheap logic after the network).

In [ ]:
W = 1 / (Xtr.var(0) + 1e-4); W = (W / W.mean()).astype(np.float32)
Wt = torch.as_tensor(W)
fit(model, Xtr, Xtr, epochs=2 if FAST else 10, bs=1024, lr=3e-3, loss_fn=lambda out, target: ((out - target) ** 2 * Wt).mean())
score = lambda m, X: ((predict(m, X, bs=8192) - X) ** 2 * W).mean(1)      # anomaly score = weighted mean squared reconstruction error
EFF = efficiency_at_fpr(score(model, Xte), y_te)
for s, e in EFF.items():
    print(f"  TPR at FPR {FPR:g}: {s:<6} {e:.4f}")

**A non-ML reference to beat.** A trigger designer's first idea: count the leptons (electrons + muons) and break ties by their total pT.
No training, almost no logic. Is your network better than this, on every signal?

In [ ]:
pt = X_te[:, 0::3]
lep = (pt[:, 1:9] > 0).sum(1) + 1e-3 * pt[:, 1:9].sum(1)
REF = efficiency_at_fpr(lep, y_te)
for s, e in REF.items():
    print(f"  lepton count, TPR at FPR {FPR:g}: {s:<6} {e:.4f}")

## 4. On the FPGA: bit-exact emulation and what hls4ml builds
`to_hls` converts the Brevitas model; `check_bit_exact` compares Brevitas with the hls4ml C++ emulation; `dense_summary` lists the multipliers per layer.
Resources (DSP, LUT) and latency in cycles come from **C-synthesis**, which the instructors run for you (`common/submit.py`). Note that the anomaly score
(the squared error, summed) is extra logic after the network: cost it too.

In [ ]:
from common.hls import to_hls, check_bit_exact, dense_summary
hm, _ = to_hls(model, output_dir="hls_prj_adc2021", in_features=57)
diff, _ = check_bit_exact(hm, model, Xte[:2000])
print(f"max |Brevitas - hls4ml| on 2,000 test events: {diff:.3g}  ({'bit-exact' if diff == 0 else 'NOT bit-exact'})")
DS = dense_summary(hm)
for r in DS:
    print(r)
MULTS = sum(r["mults_needed"] for r in DS)

In [ ]:
ROW = report("2 adc2021", (len(Xtr), len(Xva), len(Xte)), (57,),
             f"TPR@FPR{FPR:g} " + ", ".join(f"{s} {100 * e:.2f}%" for s, e in EFF.items()) + " | lepton-count ref " + ", ".join(f"{100 * e:.2f}%" for e in REF.values()), p,
             f"FPGA RF=1: {MULTS:,} multipliers (dense_summary); cycles need C-synthesis; bit-exact={diff == 0}",
             "CC BY 4.0 (Zenodo, confirmed)", extra=dict(minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.